# K-Nearest Neighbor (KNN) - Day 1 (From Scratch)

## What is KNN?

K‑Nearest Neighbor (KNN) is a simple and widely used machine learning technique for classification and regression tasks. It works by identifying the K closest data points to a given input and making predictions based on the majority class or average value of those neighbors.

**Key Characteristics:**
- Classifies data based on similarity with nearby data points
- Uses distance metrics like Euclidean distance to find nearest neighbors
- Non-parametric and instance-based learning method (no assumptions about data distribution)
- Also called a **lazy learner** because it stores the entire dataset and performs computations only at classification time

---

## How KNN Works

1. **Store all training data** (no training phase)
2. **For a new point:** Calculate distance to all training points
3. **Find K nearest neighbors** (smallest distances)
4. **Make prediction:**
   - Classification: Majority vote of K neighbors
   - Regression: Average value of K neighbors

---

## The 'K' in KNN

K is just a number that tells the algorithm how many nearby points or neighbors to look at when it makes a decision.

**Example:**
- If k = 3, looks at 3 closest fruits
- If 2 are apples and 1 is banana → predicts apple

---

## How to Choose K

| Scenario | Recommendation |
|----------|----------------|
| Small k | Sensitive to noise, can overfit |
| Large k | More stable, but may underfit |
| Noisy data | Use larger k for stability |
| Clean data | Smaller k works well |

**Common practice:** Try different k values and choose the one with best validation accuracy

---

## Distance Metrics

| Metric | Formula |
|--------|---------|
| Euclidean | √Σ(xi - yi)² |
| Manhattan | Σ|xi - yi| |
| Minkowski | (Σ|xi - yi|^p)^(1/p) |

---

## Pros and Cons

**Pros:**
- Simple and intuitive
- No training phase
- Works with any number of classes
- Easy to add new data

**Cons:**
- Slow for large datasets (O(n) per prediction)
- Sensitive to irrelevant features
- Requires feature scaling
- Memory intensive

---

## One-Line Summary

**KNN predicts based on majority vote of K closest data points using distance metrics like Euclidean distance.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print("="*50)
print("KNN DAY 1 - FROM SCRATCH")
print("="*50)

## KNN Implementation from Scratch

In [ ]:
class KNNFromScratch:
    """
    K-Nearest Neighbors implementation from scratch
    """
    
    def __init__(self, k=3, distance_metric='euclidean'):
        self.k = k
        self.distance_metric = distance_metric
        self.X_train = None
        self.y_train = None
    
    def _euclidean_distance(self, x1, x2):
        """Calculate Euclidean distance between two points"""
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    def _manhattan_distance(self, x1, x2):
        """Calculate Manhattan distance between two points"""
        return np.sum(np.abs(x1 - x2))
    
    def _minkowski_distance(self, x1, x2, p=3):
        """Calculate Minkowski distance between two points"""
        return np.sum(np.abs(x1 - x2) ** p) ** (1/p)
    
    def _distance(self, x1, x2):
        if self.distance_metric == 'euclidean':
            return self._euclidean_distance(x1, x2)
        elif self.distance_metric == 'manhattan':
            return self._manhattan_distance(x1, x2)
        elif self.distance_metric == 'minkowski':
            return self._minkowski_distance(x1, x2)
        else:
            raise ValueError("Unknown distance metric")
    
    def fit(self, X, y):
        """Store training data (lazy learning)"""
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        print(f"Stored {len(self.X_train)} training samples")
    
    def predict(self, X):
        """Predict class for each sample in X"""
        X = np.array(X)
        predictions = []
        
        for x in X:
            # Calculate distances to all training points
            distances = [self._distance(x, x_train) for x_train in self.X_train]
            
            # Get k nearest neighbors
            k_indices = np.argsort(distances)[:self.k]
            k_labels = self.y_train[k_indices]
            
            # Majority vote
            most_common = Counter(k_labels).most_common(1)[0][0]
            predictions.append(most_common)
        
        return np.array(predictions)
    
    def predict_proba(self, X):
        """Predict class probabilities for each sample"""
        X = np.array(X)
        probabilities = []
        
        for x in X:
            distances = [self._distance(x, x_train) for x_train in self.X_train]
            k_indices = np.argsort(distances)[:self.k]
            k_labels = self.y_train[k_indices]
            
            # Calculate probabilities
            counts = Counter(k_labels)
            probs = {label: count/self.k for label, count in counts.items()}
            probabilities.append(probs)
        
        return probabilities
    
    def score(self, X, y):
        """Calculate accuracy"""
        predictions = self.predict(X)
        return np.mean(predictions == np.array(y))

In [ ]:
# Create simple dataset
X = np.array([[2, 3], [3, 4], [1, 1], [2, 1], [4, 5], [5, 6], [1, 2], [3, 2]])
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print("Dataset:")
for i in range(len(X)):
    label = "Class 0" if y[i] == 0 else "Class 1"
    print(f"Point {X[i]} -> {label}")

In [ ]:
# Train KNN
knn = KNNFromScratch(k=3)
knn.fit(X, y)

print(f"\nModel trained with k={knn.k}")
print(f"Distance metric: {knn.distance_metric}")

In [ ]:
# Predict on training data
predictions = knn.predict(X)
accuracy = knn.score(X, y)

print("\nPredictions:")
for i in range(len(X)):
    print(f"Point {X[i]} -> Actual: {y[i]}, Predicted: {predictions[i]}")

print(f"\nAccuracy: {accuracy*100:.2f}%")

In [ ]:
# Test new points
new_points = np.array([[2.5, 2.5], [4.5, 5.5], [1.5, 1.5]])

print("\nPredictions for new points:")
for x in new_points:
    pred = knn.predict([x])[0]
    label = "Class 0" if pred == 0 else "Class 1"
    print(f"Point {x} -> {label}")

In [ ]:
# Visualize KNN decision boundary
def plot_knn_boundary(knn, X, y):
    plt.figure(figsize=(8, 6))
    
    # Plot points
    colors = ['blue' if label == 0 else 'red' for label in y]
    plt.scatter(X[:, 0], X[:, 1], c=colors, s=100, edgecolors='black')
    
    # Plot decision boundary
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.2, colors=['blue', 'red'])
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(f"KNN Decision Boundary (k={knn.k})")
    plt.grid(alpha=0.3)
    plt.show()

plot_knn_boundary(knn, X, y)

In [ ]:
# Effect of different k values
k_values = [1, 3, 5, 7]

print("\nEffect of different k values:")
for k in k_values:
    knn_test = KNNFromScratch(k=k)
    knn_test.fit(X, y)
    acc = knn_test.score(X, y)
    print(f"k = {k}: Accuracy = {acc*100:.2f}%")

In [ ]:
# Day 1 Completed
print("\n" + "="*50)
print("KNN DAY 1 COMPLETED")
print("="*50)
print("Topics covered:")
print("- KNN definition and key characteristics")
print("- How KNN works (distance, K neighbors, voting)")
print("- The 'K' parameter and how to choose it")
print("- Distance metrics (Euclidean, Manhattan, Minkowski)")
print("- Pros and cons of KNN")
print("- KNN implementation from scratch")
print("- Effect of different k values")
print("="*50)"
print("\nNext: Day 2 - KNN with Scikit-Learn")